# S3 Data Quality Notebook

Ноутбук для просмотра содержимого S3, загрузки parquet-данных в единый `DataFrame` и быстрой проверки качества данных.

In [43]:
import io
import os
from typing import Iterable, Optional

import boto3
import pandas as pd
import yaml
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

load_dotenv()

True

In [44]:
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DEFAULT_BUCKET = config["storage"]["bucket"]
DEFAULT_PREFIX = config["storage"]["prefix"]
DEFAULT_SOURCE = config["source"]
DEFAULT_SYMBOL = config["symbols"][0]

DEFAULT_BUCKET, DEFAULT_PREFIX, DEFAULT_SOURCE, DEFAULT_SYMBOL

('binance-data-downloader', 'raw', 'klines', 'ADAUSDT')

In [45]:
def make_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )


s3 = make_s3_client()

In [46]:
def build_partition_prefix(
    base_prefix: str,
    source: str,
    symbol: Optional[str] = None,
    interval: Optional[str] = None,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
) -> str:
    parts = [base_prefix.strip("/"), source]

    if symbol:
        parts.append(f"symbol={symbol}")

    if interval:
        parts.append(f"interval={interval}")

    prefix = "/".join(part for part in parts if part)

    if date_from and date_to and date_from == date_to:
        prefix = f"{prefix}/date={date_from}"

    return prefix.rstrip("/") + "/"


def list_s3_objects(bucket: str, prefix: str, max_keys: Optional[int] = None) -> pd.DataFrame:
    paginator = s3.get_paginator("list_objects_v2")
    rows = []
    total = 0

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            rows.append(
                {
                    "key": obj["Key"],
                    "size_bytes": obj["Size"],
                    "last_modified": obj["LastModified"],
                }
            )
            total += 1
            if max_keys is not None and total >= max_keys:
                return pd.DataFrame(rows)

    return pd.DataFrame(rows)


def _date_from_key(key: str) -> Optional[str]:
    for part in key.split("/"):
        if part.startswith("date="):
            return part.split("=", 1)[1]
    return None


def _read_parquet_from_s3(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))


def build_df(
    bucket: str,
    base_prefix: str,
    source: str,
    symbol: Optional[str] = None,
    interval: Optional[str] = None,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
    limit_files: Optional[int] = None,
) -> pd.DataFrame:
    """Собирает единый DataFrame из parquet-файлов в S3.

    Параметры date_from/date_to работают как фильтр по партициям вида date=YYYY-MM-DD.
    """
    prefix = build_partition_prefix(
        base_prefix=base_prefix,
        source=source,
        symbol=symbol,
        interval=interval,
        date_from=date_from,
        date_to=date_to,
    )

    objects_df = list_s3_objects(bucket=bucket, prefix=prefix)
    if objects_df.empty:
        raise FileNotFoundError(f"No objects found for prefix: s3://{bucket}/{prefix}")

    objects_df = objects_df[objects_df["key"].str.endswith(".parquet")].copy()
    objects_df["partition_date"] = objects_df["key"].map(_date_from_key)

    if date_from:
        objects_df = objects_df[objects_df["partition_date"] >= date_from]

    if date_to:
        objects_df = objects_df[objects_df["partition_date"] <= date_to]

    objects_df = objects_df.sort_values(["partition_date", "key"]).reset_index(drop=True)

    if limit_files is not None:
        objects_df = objects_df.head(limit_files)


    frames = []
    for key in objects_df["key"]:
        df_part = _read_parquet_from_s3(bucket=bucket, key=key)
        df_part["source_file"] = key
        df_part["partition_date"] = _date_from_key(key)
        frames.append(df_part)

    df = pd.concat(frames, ignore_index=True)

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
        df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

    return df


In [47]:
objects = list_s3_objects(
    bucket=DEFAULT_BUCKET,
    prefix=build_partition_prefix(
        base_prefix=DEFAULT_PREFIX,
        source=DEFAULT_SOURCE,
        symbol=DEFAULT_SYMBOL,
    ),
    max_keys=10000,
)

objects.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2193 entries, 0 to 2192
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype                  
---  ------         --------------  -----                  
 0   key            2193 non-null   object                 
 1   size_bytes     2193 non-null   int64                  
 2   last_modified  2193 non-null   datetime64[ns, tzutc()]
dtypes: datetime64[ns, tzutc()](1), int64(1), object(1)
memory usage: 51.5+ KB


In [ ]:
# Создать резервную копию всех объектов из папки raw
# Копия будет создана в папке raw_backup в том же бакете.

def copy_s3_prefix(bucket: str, src_prefix: str, dst_prefix: str) -> int:
    src_prefix = src_prefix.rstrip("/") + "/"
    dst_prefix = dst_prefix.rstrip("/") + "/"

    objects = list_s3_objects(bucket=bucket, prefix=src_prefix)
    if objects.empty:
        raise FileNotFoundError(f"No objects found for prefix: s3://{bucket}/{src_prefix}")

    objects = objects[objects["key"].str.startswith(src_prefix)].copy()
    objects = objects.sort_values("key").reset_index(drop=True)
    print(f"Found {len(objects)} objects under s3://{bucket}/{src_prefix}")

    copied = 0
    for key in objects["key"]:
        dest_key = key.replace(src_prefix, dst_prefix, 1)
        s3.copy_object(
            Bucket=bucket,
            CopySource={"Bucket": bucket, "Key": key},
            Key=dest_key,
        )
        copied += 1
        if copied % 100 == 0:
            print(f"Copied {copied}/{len(objects)} objects...")

    print(f"Copied {copied} objects to s3://{bucket}/{dst_prefix}")
    return copied

backup_prefix = f"{DEFAULT_PREFIX.rstrip('/')}_backup"
copy_count = copy_s3_prefix(DEFAULT_BUCKET, DEFAULT_PREFIX, backup_prefix)
print(f"Backup complete: {copy_count} objects copied.")
print(f"Source: s3://{DEFAULT_BUCKET}/{DEFAULT_PREFIX.rstrip('/')}/")
print(f"Destination: s3://{DEFAULT_BUCKET}/{backup_prefix.rstrip('/')}/")


In [48]:
df = build_df(
    bucket=DEFAULT_BUCKET,
    base_prefix=DEFAULT_PREFIX,
    source=DEFAULT_SOURCE,
    symbol=DEFAULT_SYMBOL,
    date_from="2020-02-01",
    date_to="2026-02-01",
)

df.head()

,timestamp,open_time,close_time,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,source_file,partition_date
0,2020-02-01 00:01:00+00:00,1580515260000,1580515319999,0.05382,0.05387,0.05381,0.05381,267470.0,14401.816406,25,67058.0,3609.846191,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,2020-02-01
1,2020-02-01 00:02:00+00:00,1580515320000,1580515379999,0.05385,0.05386,0.05377,0.05377,85914.0,4625.166016,20,61800.0,3327.760010,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,2020-02-01
2,2020-02-01 00:03:00+00:00,1580515380000,1580515439999,0.05377,0.05382,0.05368,0.05378,455970.0,24504.648438,49,232076.0,12473.500000,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,2020-02-01
3,2020-02-01 00:04:00+00:00,1580515440000,1580515499999,0.05376,0.05382,0.05362,0.05368,90562.0,4861.035645,33,38391.0,2062.146484,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,2020-02-01
4,2020-02-01 00:05:00+00:00,1580515500000,1580515559999,0.05364,0.05370,0.05360,0.05364,205360.0,11014.576172,30,66367.0,3560.824219,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,2020-02-01


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3157919 entries, 0 to 3157918
Data columns (total 14 columns):
 #   Column           Dtype              
---  ------           -----              
 0   timestamp        datetime64[ns, UTC]
 1   open_time        int64              
 2   close_time       int64              
 3   open             float32            
 4   high             float32            
 5   low              float32            
 6   close            float32            
 7   volume           float32            
 8   quote_volume     float32            
 9   trades           int64              
 10  taker_buy_base   float32            
 11  taker_buy_quote  float32            
 12  source_file      object             
 13  partition_date   object             
dtypes: datetime64[ns, UTC](1), float32(8), int64(3), object(2)
memory usage: 240.9+ MB


In [ ]:
# Проверка алгоритма восстановления первой минуты из локальных parquet-файлов
from pathlib import Path
from restore_first_minute import restore_first_minute

klines_path = Path("klines_ada.parquet")
trades_path = Path("trades_ada.parquet")
restored_path = Path("klines_ada_restored.parquet")

print("Исходные данные:")
original_klines = pd.read_parquet(klines_path)
print(f"  rows={len(original_klines)}")
print(f"  first timestamp={original_klines['timestamp'].min()}")
print(f"  first open_time={original_klines['open_time'].min()}")

restored = restore_first_minute(klines_path, trades_path, restored_path)
print("\nПосле восстановления:")
print(f"  rows={len(restored)}")
print(f"  first timestamp={restored['timestamp'].min()}")
print(f"  first open_time={restored['open_time'].min()}")
print(f"  сохранено в файл: {restored_path}")

print("\nПервые строки оригинального klines:")
display(original_klines.head(5))
print("\nПервые строки восстановленного klines:")
display(restored.head(6))

missing_before = restored[restored["open_time"] < original_klines["open_time"].min()]
print(f"\nВосстановленных строк до первой оригинальной: {len(missing_before)}")
display(missing_before)


In [50]:
quality_report = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=False),
    }
).sort_values(["null_pct", "nulls"], ascending=False)

quality_report

,dtype,nulls,null_pct,unique
timestamp,"datetime64[ns, UTC]",0,0.0,3157919
open_time,int64,0,0.0,3157919
close_time,int64,0,0.0,3157919
open,float32,0,0.0,113336
high,float32,0,0.0,110975
low,float32,0,0.0,110084
close,float32,0,0.0,112913
volume,float32,0,0.0,1114627
quote_volume,float32,0,0.0,3070436
trades,int64,0,0.0,12128


In [51]:
if "timestamp" in df.columns:
    print("min timestamp:", df["timestamp"].min())
    print("max timestamp:", df["timestamp"].max())
    print("duplicates:", df["timestamp"].duplicated().sum())

df.describe(include="all").T

min timestamp: 2020-02-01 00:01:00+00:00
max timestamp: 2026-02-01 23:59:00+00:00
duplicates: 0


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
timestamp,3157919,NaN,NaN,NaN,2023-02-01 12:00:00.000003840+00:00,2020-02-01 00:01:00+00:00,2021-08-02 06:00:30+00:00,2023-02-01 12:00:00+00:00,2024-08-02 17:59:30+00:00,2026-02-01 23:59:00+00:00,NaN
open_time,3157919.0,NaN,NaN,NaN,1675252800000.000244,1580515260000.0,1627884030000.0,1675252800000.0,1722621570000.0,1769990340000.0,54696770202.124367
close_time,3157919.0,NaN,NaN,NaN,1675252859999.0,1580515319999.0,1627884089999.0,1675252859999.0,1722621629999.0,1769990399999.0,54696770202.124367
open,3157919.0,NaN,NaN,NaN,0.640274,0.01782,0.3191,0.4595,0.8374,3.1037,0.529361
high,3157919.0,NaN,NaN,NaN,0.64089,0.01788,0.3193,0.4598,0.8381,3.1052,0.529965
low,3157919.0,NaN,NaN,NaN,0.639649,0.01752,0.3189,0.4592,0.8367,3.0961,0.528747
close,3157919.0,NaN,NaN,NaN,0.640273,0.01774,0.3191,0.4595,0.8374,3.104,0.529361
volume,3157919.0,NaN,NaN,NaN,503564.90625,0.0,133748.0,269038.0,546575.0,74082920.0,904390.3125
quote_volume,3157919.0,NaN,NaN,NaN,338287.75,0.0,51217.578125,140246.578125,342771.828125,128833968.0,761878.0625
trades,3157919.0,NaN,NaN,NaN,575.993337,0.0,149.0,332.0,667.0,101913.0,948.84074


In [52]:
# Проверка непрерывного временного ряда (все минуты присутствуют)

if "timestamp" in df.columns:
    # Сортируем по времени
    df_sorted = df.sort_values("timestamp").copy()
    timestamps = pd.to_datetime(df_sorted["timestamp"])
    
    # Определяем ожидаемую частоту данных
    freq = pd.infer_freq(timestamps)
    print(f"Определённая частота: {freq}")
    
    # Создаём полный набор непрерывных временных меток
    expected_range = pd.date_range(
        start=timestamps.min(),
        end=timestamps.max(),
        freq="1min"  # Ожидаем данные по каждой минуте
    )
    
    print(f"\nОжидаемое количество записей (1 минута): {len(expected_range)}")
    print(f"Фактическое количество записей: {len(df)}")
    print(f"Разница: {len(expected_range) - len(df)} пропущенных минут")
    
    # Находим пропущенные временные метки
    actual_timestamps = set(timestamps)
    expected_timestamps = set(expected_range)
    missing_timestamps = sorted(expected_timestamps - actual_timestamps)
    
    if missing_timestamps:
        print(f"\n❌ Обнаружено пропусков: {len(missing_timestamps)}")
        print(f"Первые 10 пропущенных минут:")
        for ts in missing_timestamps[:10]:
            print(f"  {ts}")
        if len(missing_timestamps) > 10:
            print(f"  ... и ещё {len(missing_timestamps) - 10}")
    else:
        print("\n✓ Временной ряд непрерывен! Все минуты присутствуют.")

Определённая частота: min

Ожидаемое количество записей (1 минута): 3157919
Фактическое количество записей: 3157919
Разница: 0 пропущенных минут

✓ Временной ряд непрерывен! Все минуты присутствуют.


In [42]:
# Сохранить даты с пропусками в CSV

if missing_timestamps:
    # Извлекаем даты из пропущенных временных меток
    missing_dates = []
    for ts in missing_timestamps:
        # Проверяем что это первая минута дня (00:00)
        if ts.hour == 0 and ts.minute == 0:
            missing_dates.append(ts.date())
    
    # Создаём DataFrame для сохранения
    missing_dates_df = pd.DataFrame({
        'date': missing_dates,
        'timestamp': missing_timestamps[:len(missing_dates)]
    })
    
    # Сохраняем в CSV
    csv_filename = "missing_dates_ada.csv"
    missing_dates_df.to_csv(csv_filename, index=False)
    print(f"✓ Сохранено {len(missing_dates)} дат в файл '{csv_filename}'")
    print("\nДаты с пропусками:")
    print(missing_dates_df)
else:
    print("Пропусков не найдено")

Пропусков не найдено


In [16]:
# btc_missing = pd.read_csv("missing_dates.csv")
ada_missing = pd.read_csv("missing_dates_ada.csv")

In [ ]:
# Проверка пересечения дат с пропусками для BTC и ADA

btc_missing = pd.read_csv("missing_dates.csv", parse_dates=["date"])
ada_missing = pd.read_csv("missing_dates_ada.csv", parse_dates=["date"])

btc_dates = set(btc_missing["date"].dt.date)
ada_dates = set(ada_missing["date"].dt.date)

common_dates = sorted(btc_dates & ada_dates)
btc_only_dates = sorted(btc_dates - ada_dates)
ada_only_dates = sorted(ada_dates - btc_dates)

print(f"Общее количество дат в BTC: {len(btc_dates)}")
print(f"Общее количество дат в ADA: {len(ada_dates)}")
print(f"Совпадающих дат: {len(common_dates)}")
print(f"Дат только в BTC: {len(btc_only_dates)}")
print(f"Дат только в ADA: {len(ada_only_dates)}")

print("\nПример дат только в BTC:")
print(btc_only_dates[:10])
print("\nПример дат только в ADA:")
print(ada_only_dates[:10])


In [24]:
# Проверка содержимого раздела trades в S3

trades_objects = list_s3_objects(
    bucket=DEFAULT_BUCKET,
    prefix=build_partition_prefix(
        base_prefix=DEFAULT_PREFIX,
        source="trades",
        symbol=DEFAULT_SYMBOL,
    ),
    max_keys=10000,
)

print(f"Найдено объектов в разделе trades: {len(trades_objects)}")
if not trades_objects.empty:
    print("Примеры ключей:")
    for key in trades_objects["key"].head(10):
        print(f"  {key}")
    
    # Группировка по датам
    trades_objects["partition_date"] = trades_objects["key"].map(_date_from_key)
    date_counts = trades_objects.groupby("partition_date").size().sort_index()
    print(f"\nДаты с файлами trades: {len(date_counts)}")
    print("Примеры:")
    for date, count in date_counts.head(10).items():
        print(f"  {date}: {count} файл(ов)")
    
    if len(date_counts) > 10:
        print(f"  ... и ещё {len(date_counts) - 10} дат")
else:
    print("Раздел trades пустой")

Найдено объектов в разделе trades: 2193
Примеры ключей:
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-01/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-02/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-03/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-04/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-05/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-06/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-07/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-08/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-09/data.parquet
  raw_backup/trades/symbol=ADAUSDT/date=2020-02-10/data.parquet

Даты с файлами trades: 2193
Примеры:
  2020-02-01: 1 файл(ов)
  2020-02-02: 1 файл(ов)
  2020-02-03: 1 файл(ов)
  2020-02-04: 1 файл(ов)
  2020-02-05: 1 файл(ов)
  2020-02-06: 1 файл(ов)
  2020-02-07: 1 файл(ов)
  2020-02-08: 1 файл(ов)
  2020-02-09: 1 файл(ов)
  2020-02-10: 1 файл(ов)
  ... и ещё 2183

In [2]:
import pandas as pd
df = pd.read_parquet(r"C:\projects\binance-dowloader-3.0\klines_ada_restored.parquet")
df

,timestamp,open_time,close_time,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
0,2020-02-02 00:00:00+00:00,1580601600000,1580601659999,0.05615,0.05623,0.05615,0.05617,58146.0,3268.307129,24,54832.0,3082.156250
1,2020-02-02 00:01:00+00:00,1580601660000,1580601719999,0.05613,0.05623,0.05608,0.05608,229922.0,12902.596680,41,21260.0,1195.124634
2,2020-02-02 00:02:00+00:00,1580601720000,1580601779999,0.05615,0.05626,0.05610,0.05615,255347.0,14349.841797,42,179892.0,10113.105469
3,2020-02-02 00:03:00+00:00,1580601780000,1580601839999,0.05615,0.05620,0.05610,0.05613,86645.0,4865.781250,38,36242.0,2036.588135
4,2020-02-02 00:04:00+00:00,1580601840000,1580601899999,0.05614,0.05620,0.05607,0.05616,127403.0,7151.403320,55,58831.0,3304.817871
...,...,...,...,...,...,...,...,...,...,...,...,...
1435,2020-02-02 23:55:00+00:00,1580687700000,1580687759999,0.05591,0.05591,0.05583,0.05583,209647.0,11709.338867,19,25565.0,1427.549561
1436,2020-02-02 23:56:00+00:00,1580687760000,1580687819999,0.05585,0.05587,0.05575,0.05587,457749.0,25538.960938,35,19909.0,1112.192139
1437,2020-02-02 23:57:00+00:00,1580687820000,1580687879999,0.05585,0.05587,0.05585,0.05587,54462.0,3042.681152,9,49924.0,2789.233887
1438,2020-02-02 23:58:00+00:00,1580687880000,1580687939999,0.05583,0.05585,0.05583,0.05585,238289.0,13306.222656,25,138470.0,7732.330078


In [25]:
# Проверка непрерывности дат в разделе trades

if 'trades_objects' in globals() and not trades_objects.empty:
    # Получаем уникальные даты из файлов trades
    trades_dates = pd.to_datetime(trades_objects["partition_date"]).dt.date.unique()
    trades_dates = sorted(trades_dates)
    
    if trades_dates:
        # Определяем диапазон дат
        start_date = min(trades_dates)
        end_date = max(trades_dates)
        
        # Создаем полный диапазон дат
        expected_dates = pd.date_range(start=start_date, end=end_date, freq='D').date
        
        print(f"Диапазон дат в trades: от {start_date} до {end_date}")
        print(f"Ожидаемое количество дат: {len(expected_dates)}")
        print(f"Фактическое количество дат: {len(trades_dates)}")
        
        # Находим пропущенные даты
        actual_dates_set = set(trades_dates)
        expected_dates_set = set(expected_dates)
        missing_dates = sorted(expected_dates_set - actual_dates_set)
        
        if missing_dates:
            print(f"\n❌ Обнаружено пропусков: {len(missing_dates)} дат")
            print("Пропущенные даты:")
            for date in missing_dates[:20]:  # Показываем первые 20
                print(f"  {date}")
            if len(missing_dates) > 20:
                print(f"  ... и ещё {len(missing_dates) - 20} дат")
        else:
            print("\n✓ Диапазон дат непрерывен! Все даты присутствуют.")
    else:
        print("Не удалось определить даты из файлов trades")
else:
    print("Сначала выполните ячейку проверки содержимого раздела trades")

Диапазон дат в trades: от 2020-02-01 до 2026-02-01
Ожидаемое количество дат: 2193
Фактическое количество дат: 2193

✓ Диапазон дат непрерывен! Все даты присутствуют.
